# 20. Uji Signifikansi Perbandingan Konfigurasi Fitur Multiclass

Notebook ini digunakan untuk menjalankan uji signifikansi kontribusi fitur menggunakan paired bootstrap. Uji dilakukan pada prediksi XGBoost untuk setiap konfigurasi fitur agar perbandingan berfokus pada pengaruh fitur, bukan perbedaan model.

Uji yang digunakan:

- Paired bootstrap resampling pada data testing yang sama.
- Model dibuat tetap, yaitu XGBoost, untuk seluruh konfigurasi fitur.
- Metrik yang diuji adalah selisih accuracy antara konfigurasi dasar dan konfigurasi pembanding.
- Peningkatan dianggap signifikan apabila interval kepercayaan 95% tidak melewati nilai nol.

In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "bootstrap_feature_comparison_multiclass.py"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

print("Project root:", PROJECT_ROOT.resolve())
print("Script:", SCRIPT_PATH.resolve())
print("Artifacts:", ARTIFACT_DIR.resolve())

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(SCRIPT_PATH)

In [ ]:
subprocess.run([sys.executable, str(SCRIPT_PATH)], check=True)

In [ ]:
report_path = ARTIFACT_DIR / "bootstrap_multiclass_xgboost_report.txt"
print(report_path.read_text(encoding="utf-8"))

In [ ]:
import pandas as pd

summary_path = ARTIFACT_DIR / "bootstrap_multiclass_xgboost_feature_summary.csv"
summary_df = pd.read_csv(summary_path)
display(summary_df)

In [ ]:
ablation_path = ARTIFACT_DIR / "bootstrap_multiclass_xgboost_feature_ablation.csv"
ablation_df = pd.read_csv(ablation_path)
ablation_df = ablation_df.sort_values("accuracy_diff", ascending=False)
display(ablation_df[[
    "feature_added",
    "interpretation",
    "baseline",
    "baseline_accuracy",
    "comparison",
    "comparison_accuracy",
    "accuracy_diff",
    "ci_low",
    "ci_high",
    "p_bootstrap",
    "significant",
]])

Interpretasi utama dari hasil uji signifikansi:

- Penambahan FFT memberikan peningkatan accuracy paling besar pada XGBoost, terutama ketika ditambahkan pada konfigurasi IQA only.
- CLIP juga memberikan kontribusi signifikan ketika ditambahkan pada IQA only dan FFT only.
- NR-IQA lebih tepat diposisikan sebagai fitur pelengkap, karena penambahannya pada konfigurasi FFT + CLIP tidak signifikan berdasarkan interval kepercayaan 95%.
- IQA + FFT + CLIP adalah konfigurasi terbaik secara numerik, sedangkan fitur dengan kontribusi paling kuat secara statistik pada XGBoost adalah FFT.